# 02 — Build and Run an Agent

This notebook walks you through interacting with a deployed agent or creating one from scratch.
Run cells top-to-bottom. Change `AGENT_NAME` below to work with any registered agent.

## Configuration

In [ ]:
# Change this to work with a different agent
AGENT_NAME = "code-helper"

## Install Dependencies

In [ ]:
%pip install azure-ai-agents azure-identity pydantic-settings python-dotenv -q
# Or if using uv from a terminal: uv sync

## Connect to Foundry

In [ ]:
from dotenv import load_dotenv
import os
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

# Load connection string from .env
load_dotenv()
CONNECTION_STRING = os.environ["FOUNDRY_PROJECT_CONNECTION_STRING"]

client = AgentsClient(
    endpoint=CONNECTION_STRING,
    credential=DefaultAzureCredential()
)
print("✓ Connected to Azure AI Foundry")

## Step 1: Get or Deploy the Agent

Uses the existing deployed agent if found, otherwise deploys it automatically via the registry.

In [4]:
from agents.registry import REGISTRY

# Always deploy/update the agent via the factory.
# This ensures the latest instructions + tools are pushed to Foundry.
entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()
agent = entry.factory(config)
print(f"✓ Agent ready: {agent.name} (id: {agent.id}, model: {agent.model})")

## Step 2: Create a Thread and Send a Message

In [ ]:
# Create a conversation thread
thread = client.threads.create()
print(f"✓ Thread created: {thread.id}")

# Post a message — change the content to test different prompts
message = client.messages.create(
    thread_id=thread.id,
    role="user",
    content="Hello! What can you help me with?",
)
print(f"✓ Message posted: {message.id}")

## Step 3: Run the Agent and Get a Response

In [ ]:
import importlib, json
from azure.ai.agents.models import MessageRole, RunStatus

# Load and register the agent's tool functions so the SDK can auto-execute them
module_name = AGENT_NAME.replace("-", "_")
tools_module = importlib.import_module(f"agents.{module_name}.tools")
if hasattr(tools_module, "TOOLS") and tools_module.TOOLS:
    for tool in tools_module.TOOLS:
        client.enable_auto_function_calls(tool)
    print(f"✓ Registered {len(tools_module.TOOLS)} tool(s) for auto-execution")

# --- Continuation loop ---
MAX_CONTINUATIONS = 10
continuation = 0
consecutive_failures = 0
response = "(no response)"

def get_run_response(thread_id, run_id):
    """Get ALL assistant text responses from a specific run, concatenated."""
    msgs = list(client.messages.list(thread_id=thread_id, run_id=run_id))
    texts = []
    for msg in msgs:
        if msg.role == "assistant":
            for part in msg.content:
                if hasattr(part, "text") and part.text and part.text.value.strip():
                    texts.append(part.text.value.strip())
    texts.reverse()
    return "\n\n".join(texts) if texts else None

def log_tool_calls(thread_id, run_id):
    """Print a compact summary of tool calls made during this run."""
    for step in client.run_steps.list(thread_id=thread_id, run_id=run_id):
        if step.type == "tool_calls" and hasattr(step, "step_details"):
            for tc in step.step_details.tool_calls:
                raw = tc.as_dict() if hasattr(tc, "as_dict") else {}
                fn = raw.get("function", raw.get("code_interpreter", {}))
                name = fn.get("name", step.type)
                args = fn.get("arguments", "")
                status = step.status
                if len(str(args)) > 200:
                    args = str(args)[:200] + "..."
                print(f"    [{status}] {name}({args})")

def get_failed_error_hint(run):
    """Build a targeted recovery prompt based on the error type."""
    err = ""
    if run.last_error:
        err = f"{run.last_error.code} — {run.last_error.message}"
    if "422" in err:
        return (
            "The previous tool call failed with 422. If creating/updating a file, "
            "fetch its current sha first, then retry with the sha."
        )
    if "404" in err:
        return (
            "The previous tool call got 404. The resource may not exist yet — "
            "create it directly or skip and continue."
        )
    return "The previous tool call failed. Adjust and retry, or skip and continue."

while True:
    run = client.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id,
        max_completion_tokens=16384,
        max_prompt_tokens=128000,
    )

    continuation += 1
    status_str = str(run.status)
    print(f"\n{'='*60}")
    print(f"Run #{continuation} — status: {status_str}")

    if run.status == RunStatus.FAILED:
        consecutive_failures += 1
        if run.last_error:
            print(f"  ✗ Error: {run.last_error.code} — {run.last_error.message}")
        if consecutive_failures >= 3:
            print(f"\n✗ Giving up after {consecutive_failures} consecutive failures")
            break
        hint = get_failed_error_hint(run)
        print(f"  → Retrying ({consecutive_failures}/3)...")
        client.messages.create(thread_id=thread.id, role="user", content=hint)
        continue

    consecutive_failures = 0

    if run.status in (RunStatus.CANCELLED, RunStatus.EXPIRED):
        print(f"  ✗ Run ended: {status_str}")
        break

    run_response = get_run_response(thread.id, run.id)
    if run_response:
        response = run_response

    steps = list(client.run_steps.list(thread_id=thread.id, run_id=run.id))
    tool_steps = [s for s in steps if s.type == "tool_calls"]
    print(f"  Tool-call steps: {len(tool_steps)} | Total steps: {len(steps)}")
    log_tool_calls(thread.id, run.id)
    print(f"  Response: {response[:300]}...")

    if "incomplete" in status_str.lower():
        reason = ""
        if hasattr(run, "incomplete_details") and run.incomplete_details:
            reason = f" ({run.incomplete_details})"
        print(f"  ⚠ Incomplete{reason} — continuing")
    elif run.status == RunStatus.COMPLETED:
        # Single-turn: break after the first completed run
        print(f"\n✓ Agent done after {continuation} run(s)")
        break

    if continuation >= MAX_CONTINUATIONS:
        print(f"\n⚠ Reached max runs ({MAX_CONTINUATIONS})")
        break

    print("  → Continuing...")
    client.messages.create(
        thread_id=thread.id,
        role="user",
        content="Continue where you left off.",
    )

print(f"\n{'='*60}")
print(f"Final response:\n{response}")

In [ ]:
# Debug: dump ALL run steps to see what the agent actually did
try:
    for step in client.run_steps.list(thread_id=thread.id, run_id=run.id):
        print(f"--- Step {step.id} | type: {step.type} | status: {step.status} ---")
        if hasattr(step, "step_details"):
            details = step.step_details
            if hasattr(details, "tool_calls"):
                for tc in details.tool_calls:
                    raw = tc.as_dict() if hasattr(tc, "as_dict") else str(tc)
                    print(json.dumps(raw, indent=2, default=str)[:3000])
            elif hasattr(details, "as_dict"):
                print(json.dumps(details.as_dict(), indent=2, default=str)[:2000])
        print()
except Exception as e:
    print(f"⚠ Could not retrieve run steps: {e}")

## Step 4: Cleanup

Delete the thread. The deployed agent is kept for future use.

In [ ]:
# Clean up the thread (keeps the deployed agent)
client.threads.delete(thread.id)
print("✓ Thread deleted (agent kept for future use)")

## Next Steps

- Change the message content in Step 2 to test different prompts
- Try a different agent: set `AGENT_NAME = "doc-assistant"` and re-run
- Add custom tools — see the [Custom Tools Guide](../docs/custom-tools-guide.md) for a full walkthrough
- See **03_scaffold_agent.ipynb** to create a brand new agent